Vamos a introducir algunas anotaciones sobre la arquitectura que se va a tomar en principio para entender mejor el contexto de las decisiones en este notebook.

Las explicaciones detalladas se encuentran en la memoria.

El clustering va a ser calculado con todas las utterances del corpus del subreddit de overwatch, ya que si lo realizamos solo en los titulos+descripcion de los posts es casi seguro que perderemos muchisima información con clusters de temáticas recurrentes entre la comunidad.

El problema que se presenta aqui es que existen muchos posts donde el cluster en común no es una contextualización perteneciente a una caracteristica del juego.

Replies como "this is awesome" o "this was so good" no aportan ningún valor a la hora de clusterizar lo que gusta o no gusta del juego. 

Para solucionar esto vamos a afrontarlo de dos maneras:

Primero vamos a confiar en que la clusterizacion no supervisada, va a agrupar todas estas utterances en clusters comunes (cluster de opinion, cluster de usuario dando animo a otro usuario), esto nos interesa ya que los clusters que no aportan valor en la toma de decisiones de negocio van a tender a agruparse ya de por si.

Luego deberíamos aplicar un filtrado a todos los clusters, donde apliquemos un analisis de términos pertenecientes al contexto especifico de nuestro target, para descartar los clusters de texto que sean agrupaciones de contexto común lingüistico.

Por último, necesitamos que las clusterizaciones sean entendibles por seres humanos, y no sean agrupaciones de embeddings incomprensibles. Para esta tarea un posible enfoque sería guardar los ids de las uterancias que conforman cada cluster, para luego hacer una agrupación de los textos que han quedado mas centralizados en un cluster especifico, para así poder hacer un analisis c-TF-IDF (class-bassed TF-IDF) que nos devuelva los términos mas carácteristicos de esos textos.

Además de esto, las nuevas tecnologías incluyen resumenes generados por LLMs, donde se puede alimentar a un modelo generativo con un sampleo reducido de los textos mas centralizados de ese cluster, asi como su TD-IDF, para que genere un resumen de ese cluster. Esto aporta mucho valor a la hora de presentar esta información a otras personas. Además de que es realmente interesante para proyectos en los que el target sea externo a nuestra empresa, y no dispongamos de la información contextual en mente como tendríamos para nuestra propia empresa.

En resumen. Los pasos que queremos realizar en esta parte de la herramienta son:

- Clusterizacion de todo tipo de textos
- Hidratación de clusters con TD-IDF y LLMs para sacar los topics dentro de cada cluster
- Filtrado de clusters según la relevancia que tengan en nuestro contexto (TD-IDF global)

Como anotación final, es muy importante conservar en todo momento los utterance_id, ya que para el sentiment analysis los necesitaremos 

Tenemos la suerte de que ya existe un modelo creado con gran parte de nuestro pipeline, que es el de BERTopic.

Este modelo hace de forma automatizada la extracción de embedings para la clusterizacion, la clusterización en si (HDBSCAN) con reduccion de diensionalidad (UMAP) y luego TD-IDF para cada cluster. Además tiene soporte nativo para "representation models" que incluye la generación de etiquetas via LLM. 

La única parte que nos quedaría por hacer por nuestra cuenta es la clasificacion de clusters relevantes.

Primero vamos a comprobar que los ficheros _truncated se corresponden por posts, y no hay replies de posts que no estan dentro de nuestra ventana temporal a analizar

In [1]:
import json

CONV_TRUNCATED_PATH = "D:\\TFM\\data\\Overwatch.corpus\\conversations_truncated.json"
UTT_TRUNCATED_PATH = "D:\\TFM\\data\\Overwatch.corpus\\utterances_truncated.jsonl"

with open(CONV_TRUNCATED_PATH, "r", encoding="utf-8") as f:
    sampled_ids = set(json.load(f).keys())

n_total = 0
n_orphan = 0
with open(UTT_TRUNCATED_PATH, "r", encoding="utf-8") as f:
    for line in f:
        n_total += 1
        utt = json.loads(line)
        if utt.get("root") not in sampled_ids:
            n_orphan += 1

print(f"Total utterances: {n_total:,}")
print(f"Utterances con root fuera de la muestra (huérfanas): {n_orphan:,}")
print(f"¿Todas pertenecen a la muestra?: {n_orphan == 0}")

Total utterances: 3,975,655
Utterances con root fuera de la muestra (huérfanas): 0
¿Todas pertenecen a la muestra?: True


Ahora vamos a construir nuestro data set agrupado con posts (title + description) y replies.

In [2]:
import json
import pandas as pd
import re
import html

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r"\[deleted\]", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\[removed\]", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

# 1. Metadata de conversaciones (para recuperar título y datos a nivel de post)
with open(CONV_TRUNCATED_PATH, "r", encoding="utf-8") as f:
    conv_meta = json.load(f)  # {root_id: {title, num_comments, domain, timestamp, subreddit, gilded, ...}}

# 2. Recorrer TODAS las utterances (posts + comentarios), conservando todas las features
rows = []
with open(UTT_TRUNCATED_PATH, "r", encoding="utf-8") as f:
    for line in f:
        utt = json.loads(line)
        meta = utt.get("meta", {}) or {}
        is_post = utt.get("reply_to") is None
        root = utt.get("root")
        parent_meta = conv_meta.get(root, {})

        raw_text = utt.get("text", "")

        # Texto para clustering: posts llevan título+cuerpo, comentarios solo su propio texto
        if is_post:
            text_for_clustering = f"{parent_meta.get('title', '')} {raw_text}".strip()
        else:
            text_for_clustering = raw_text

        rows.append({
            "id": utt.get("id"),
            "root": root,
            "reply_to": utt.get("reply_to"),
            "is_post": is_post,
            "user": utt.get("user"),
            "raw_text": raw_text,
            "text_for_clustering": text_for_clustering,
            "post_title": parent_meta.get("title"),          # título del post padre, útil como referencia/dashboard
            "timestamp": utt.get("timestamp"),
            "score": meta.get("score"),
            "top_level_comment": meta.get("top_level_comment"),
            "gilded": meta.get("gilded"),
            "gildings": meta.get("gildings"),
            "subreddit": meta.get("subreddit"),
            "stickied": meta.get("stickied"),
            "permalink": meta.get("permalink"),
            "author_flair_text": meta.get("author_flair_text"),
            "post_num_comments": parent_meta.get("num_comments"),
            "post_domain": parent_meta.get("domain"),
        })

df_clustering = pd.DataFrame(rows)
print(f"Total filas: {len(df_clustering):,}")
print(f"Posts: {df_clustering['is_post'].sum():,} | Comentarios: {(~df_clustering['is_post']).sum():,}")

df_clustering["clean_text"] = df_clustering["text_for_clustering"].apply(clean_text)

vacios = (df_clustering["clean_text"].str.strip() == "").sum()
print(f"Filas con clean_text vacío: {vacios:,}")

df_clustering.to_parquet("D:\\TFM\\data\\Overwatch.corpus\\overwatch_clustering_dataset.parquet", index=False)
print("Guardado.")

Total filas: 3,975,655
Posts: 300,000 | Comentarios: 3,675,655
Filas con clean_text vacío: 151,685
Guardado.


Ahora vamos a descartar los textos sin texto, y ver cuantos textos hay con poco texto (sin descartar estos ultimos todavía)

In [1]:
import pandas as pd

df = pd.read_parquet("D:\\TFM\\data\\Overwatch.corpus\\overwatch_clustering_dataset.parquet")

# descartar vacíos
df = df[df["clean_text"].str.strip() != ""].copy()
print(f"Tras quitar vacíos: {len(df):,}")

# cuantificar (no descartar todavía) filas de muy poca señal
df["n_words"] = df["clean_text"].str.split().str.len()
print(df["n_words"].describe())
print(f"\nFilas con <= 2 palabras: {(df['n_words'] <= 2).sum():,}")
print(f"Filas con <= 4 palabras: {(df['n_words'] <= 4).sum():,}")

Tras quitar vacíos: 3,823,970
count    3.823970e+06
mean     3.375828e+01
std      5.906520e+01
min      1.000000e+00
25%      8.000000e+00
50%      1.700000e+01
75%      3.800000e+01
max      6.510000e+03
Name: n_words, dtype: float64

Filas con <= 2 palabras: 227,461
Filas con <= 4 palabras: 503,431


Al disponer de una cantidad tan grando de filas (3.8 M), vamos a realizar un benchmark pequeño para ver como va a funcionar nuestra computadora a la hora de usar BERTopic para la clusterizacion, y ver si es conveniente truncar algo mas nuestros datos.

In [2]:
from sentence_transformers import SentenceTransformer
import time
import torch

print("CUDA disponible:", torch.cuda.is_available())

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")

sample_texts = df["clean_text"].sample(5000, random_state=42).tolist()

t0 = time.time()
_ = model.encode(sample_texts, batch_size=128, show_progress_bar=True)
elapsed = time.time() - t0

docs_per_sec = len(sample_texts) / elapsed
print(f"\nThroughput: {docs_per_sec:.1f} docs/seg")
print(f"Estimación para el dataset completo ({len(df):,} filas): {len(df)/docs_per_sec/60:.1f} minutos")

CUDA disponible: True


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\KSchool\TodosEnvs\dl-venv2\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LIGHT\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]


Throughput: 693.1 docs/seg
Estimación para el dataset completo (3,823,970 filas): 92.0 minutos


92 minutos. Parece un tiempo de ejecución asumible.

Para blindar el proceso de algún fallo sistematico, vamos a procesar y guardar resultados por chunks, no en una sola llamada bloqueante. 

Además evitamos sobrecargar la RAM al acumular el array final. Guardar a disco progresivamente evita tener que mantenerlo todo en memoria simultáneamente antes de persistirlo.

Primero vamos a sacar los embeddings usando un modelo basado en BERT con self-attention llamado "MiniLM-L6-v2" es un modelo L6 con 6 capas de transformers para vectorizar cada token.

In [3]:
import numpy as np
import os
import time
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

texts = df["clean_text"].tolist()
ids = df["id"].tolist()  # para poder reconstruir la correspondencia embedding <-> fila después

CHUNK_SIZE = 200_000
OUT_DIR = "D:\\TFM\\data\\Overwatch.corpus\\embeddings_chunks"
os.makedirs(OUT_DIR, exist_ok=True)

n_total = len(texts)
n_chunks = (n_total // CHUNK_SIZE) + 1

for i in range(n_chunks):
    start = i * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, n_total)
    if start >= n_total:
        break

    chunk_path = f"{OUT_DIR}\\chunk_{i:03d}.npy"
    ids_path = f"{OUT_DIR}\\chunk_{i:03d}_ids.npy"

    if os.path.exists(chunk_path):
        print(f"Chunk {i} ya existe, saltando...")
        continue

    t0 = time.time()
    chunk_texts = texts[start:end]
    chunk_ids = ids[start:end]

    chunk_embeddings = model.encode(
        chunk_texts, batch_size=128, show_progress_bar=False, convert_to_numpy=True
    )

    np.save(chunk_path, chunk_embeddings)
    np.save(ids_path, np.array(chunk_ids))

    elapsed = time.time() - t0
    print(f"Chunk {i+1}/{n_chunks} ({start:,}-{end:,}) -> {elapsed:.1f}s "
          f"({len(chunk_texts)/elapsed:.1f} docs/seg)")

print("\nCompletado.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunk 1/20 (0-200,000) -> 190.9s (1047.6 docs/seg)
Chunk 2/20 (200,000-400,000) -> 182.5s (1096.2 docs/seg)
Chunk 3/20 (400,000-600,000) -> 177.2s (1128.5 docs/seg)
Chunk 4/20 (600,000-800,000) -> 186.2s (1074.1 docs/seg)
Chunk 5/20 (800,000-1,000,000) -> 179.8s (1112.4 docs/seg)
Chunk 6/20 (1,000,000-1,200,000) -> 177.4s (1127.4 docs/seg)
Chunk 7/20 (1,200,000-1,400,000) -> 165.3s (1210.3 docs/seg)
Chunk 8/20 (1,400,000-1,600,000) -> 175.0s (1142.8 docs/seg)
Chunk 9/20 (1,600,000-1,800,000) -> 165.8s (1206.2 docs/seg)
Chunk 10/20 (1,800,000-2,000,000) -> 164.9s (1212.9 docs/seg)
Chunk 11/20 (2,000,000-2,200,000) -> 168.4s (1187.6 docs/seg)
Chunk 12/20 (2,200,000-2,400,000) -> 164.4s (1216.4 docs/seg)
Chunk 13/20 (2,400,000-2,600,000) -> 177.1s (1129.5 docs/seg)
Chunk 14/20 (2,600,000-2,800,000) -> 171.9s (1163.8 docs/seg)
Chunk 15/20 (2,800,000-3,000,000) -> 167.7s (1192.3 docs/seg)
Chunk 16/20 (3,000,000-3,200,000) -> 167.1s (1196.9 docs/seg)
Chunk 17/20 (3,200,000-3,400,000) -> 163.

recostrucción final

In [4]:
import glob

chunk_files = sorted(glob.glob(f"{OUT_DIR}\\chunk_*[!s].npy"))  # evita los *_ids.npy
id_files = sorted(glob.glob(f"{OUT_DIR}\\chunk_*_ids.npy"))

all_embeddings = np.concatenate([np.load(f) for f in chunk_files], axis=0)
all_ids = np.concatenate([np.load(f) for f in id_files], axis=0)

print(f"Embeddings totales: {all_embeddings.shape}")
print(f"Ids totales: {all_ids.shape}")

np.save("D:\\TFM\\data\\Overwatch.corpus\\overwatch_embeddings_full.npy", all_embeddings)
np.save("D:\\TFM\\data\\Overwatch.corpus\\overwatch_embeddings_ids.npy", all_ids)

Embeddings totales: (3823970, 384)
Ids totales: (3823970,)


Okay ya tenemos nuestros embeddings guardados en el archivo .npy (archivo de matrices numpy)

Ahora antes de lanzar UMAP+HDBSCAN directamente sobre los 3.8 M de textos, hay que replantear la estratégia.

Generar embeddings es inferencia pura. Cada documento se procesa de forma independiente, asi que escala linealmente y se puede paralelizar/trocear sin problema.

Sin embargo UMAP y HDBSCAN funcionan de manera diferente. Ambos analisis necesitan razonar sobre las relaciones entre todos los puntos a la vez. Con los 16GB de ram que tenemos disponibles, nos encontramos una limitación que tenemos que solventar.

La solución estandar recomendada en la documentación de BERTopic es entrenar (fit) sobre una muestra representativa, y luego asignar (transform) el resto.

- 1. Ajustamos el modelo completo (UMAP + HDBSCAN + c-TF-IDF) sobre una muestra manejable (300.000 - 400.000), esto define la estructura de los clusters/topics
- 2. Usamos topic_model.transform() sobre el resto de los 3.8M. Esto es mucho mas barato computacionalmente, por que solo necesita proyectar cada nuevo punto contra la estructura ya aprendida (UMAP entrenado + predicción aproximada de HDBSCAN), no recalcula la estructura desde cero. 

Aqui es recomendable, si se quiere reproducir el notebook, reiniciar el kernel, ya que tiene muchas variables cargadas que no vamos a usar, y ocupan espacio en memoria que necesitamos para la reduccion de dimensionalidad y la clusterización.

No vamos a perder los embeddings ya que los tenemos guardados en 

data\Overwatch.corpus\overwatch_embeddings_full.npy

data\Overwatch.corpus\overwatch_embeddings_ids.npy

data\Overwatch.corpus\overwatch_clustering_dataset.parquet

Una vez reiniciamos el kernel, volvemos a cargar las variables necesarias

In [1]:
import numpy as np
import pandas as pd
import psutil

embeddings = np.load("D:\\TFM\\data\\Overwatch.corpus\\overwatch_embeddings_full.npy")
ids = np.load("D:\\TFM\\data\\Overwatch.corpus\\overwatch_embeddings_ids.npy", allow_pickle=True)

# carga solo las columnas que realmente necesitas para el fit (id + texto), no todas las features
df_texts = pd.read_parquet(
    "D:\\TFM\\data\\Overwatch.corpus\\overwatch_clustering_dataset.parquet",
    columns=["id", "raw_text"]
)
df_texts = df_texts.set_index("id").loc[ids].reset_index()

print(f"RAM tras carga mínima: {psutil.virtual_memory().available / (1024**3):.1f} GB disponibles")

RAM tras carga mínima: 3.6 GB disponibles


Hemos instalado la librería psutil para hacer el entrenamiento monitoreando nuestra RAM, para saber si podemos hacer un entrenamiento mas extensivo o estamos al límite.

In [2]:
print(f"RAM disponible ahora mismo: {psutil.virtual_memory().available / (1024**3):.1f} GB")

RAM disponible ahora mismo: 4.4 GB


Entrenamos con monitoreo

In [3]:
import time
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

SAMPLE_SIZE = 350_000   # tamaño conservador y ya validado como razonable; no lo subas todavía
SEED = 42

rng = np.random.RandomState(SEED)
sample_idx = rng.choice(len(embeddings), size=SAMPLE_SIZE, replace=False)
emb_sample = embeddings[sample_idx]
docs_sample = df_texts.iloc[sample_idx]["raw_text"].fillna("").tolist()

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                   low_memory=True, random_state=SEED)
hdbscan_model = HDBSCAN(min_cluster_size=200, min_samples=10, metric="euclidean",
                         cluster_selection_method="eom", prediction_data=True)

topic_model = BERTopic(embedding_model=None, umap_model=umap_model,
                        hdbscan_model=hdbscan_model, calculate_probabilities=False,
                        verbose=True)

print(f"RAM antes del fit: {psutil.virtual_memory().available / (1024**3):.1f} GB")
t0 = time.time()

topics_sample, _ = topic_model.fit_transform(docs_sample, embeddings=emb_sample)

print(f"\nTiempo total: {(time.time()-t0)/60:.1f} min")
print(f"RAM después del fit: {psutil.virtual_memory().available / (1024**3):.1f} GB")

# GUARDAR INMEDIATAMENTE - esto es lo que faltaba antes
topic_model.save(
    "D:\\TFM\\data\\Overwatch.corpus\\bertopic_model_350k",
    serialization="pickle"
)
np.save("D:\\TFM\\data\\Overwatch.corpus\\bertopic_sample_idx.npy", sample_idx)
np.save("D:\\TFM\\data\\Overwatch.corpus\\bertopic_sample_topics.npy", np.array(topics_sample))

print("\n✅ Modelo guardado en disco.")

2026-07-18 21:22:07,017 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


RAM antes del fit: 2.6 GB


2026-07-18 22:12:49,685 - BERTopic - Dimensionality - Completed ✓
2026-07-18 22:12:49,707 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-18 22:13:47,552 - BERTopic - Cluster - Completed ✓
2026-07-18 22:13:47,695 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-18 22:13:59,783 - BERTopic - Representation - Completed ✓
2026-07-18 22:14:03,123 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.



Tiempo total: 51.9 min
RAM después del fit: 6.8 GB

✅ Modelo guardado en disco.


Comprobamos resultado

In [4]:
n_topics = len(set(topics_sample)) - (1 if -1 in topics_sample else 0)
print(f"Topics encontrados: {n_topics}")
print(topic_model.get_topic_info().head(20))

Topics encontrados: 226
    Topic   Count                                  Name  \
0      -1  144364                     -1_to_the_and_you   
1       0    7646                   0_mercy_rez_her_she   
2       1    6933               1_skins_skin_boxes_loot   
3       2    6102              2_chat_report_banned_ban   
4       3    5881               3_dps_tank_healer_tanks   
5       4    4774          4_hanzo_scatter_hanzos_arrow   
6       5    3799  5_downvoted_opinion_downvotes_upvote   
7       6    3750             6_sombra_hack_sombras_her   
8       7    3375                  7_dva_mech_dvas_meka   
9       8    3241          8_dallas_seoul_london_korean   
10      9    3135              9_read_sense_post_posted   
11     10    2852       10_lucio_lucios_lucioball_speed   
12     11    2782          11_genji_genjis_deflect_dash   
13     12    2756                 12_ana_anas_her_sleep   
14     13    2722                13_mei_meis_freeze_ice   
15     14    2660               

Buenos resultados. 226 topics encontrados. Las agrupaciones de cTD-IDF y representative_docs parecen ser coherentes a primera vista.

Tambien observamos que 144364 textos (41% aprox) caen en la etiqueta -1 de ruido del BERTopic. Esto tiene sentido ya que para no perder señal informativa, decidimos dejar en el muestreo mensajes cortos que podían tener señal informativa.

De todos modos vamos a analizarlos brevemente para comprobar si se ha perdido realmente información o no en la agrupación de ruido. De ser así se podría ajustar con hiperparámetros menos estrictos.

In [5]:
noise_docs = df_texts.iloc[sample_idx][np.array(topics_sample) == -1]
print(noise_docs["raw_text"].sample(30, random_state=42).tolist())

['Either 2.5 seconds OR a much faster refuel would be fine. As of now, DM is really hard to get value out of for your team (since often you have to save it for your own butt)', 'https://gfycat.com/MintyCharmingIlsamochadegu\n\nApparently very low', "For such a huge company they really do have crappy servers. Even when they're functioning as intended, it's not great.", 'Haha nice username ', 'no it stays until damage depletes it.', 'Eh, no, to lazy to provide proof of being lazy.', 'What is this technique?', 'Yes because petitions on this subreddit have always succeeded.', 'Made a rude Gesture.', 'I’m guessing some bitch who spammed hello', 'But the reasons are already laid out, you can argue with them if you want, but like, theyre official stated reasons.', "I'm glad it's not in the sidebar. The main sub is a cesspool and the less people from here that metastasize to /r/cow the better", "I'm not sure about the specifics of implementation, though I concur that there's enough effective o

También podemos observar algunos clusters extraños, que probablemente tendremos que descartar en la fase de filtrado final que ya comentamos, vamos a explorarlos.

In [6]:
topic_model.get_topic(14)
docs_topic14 = df_texts.iloc[sample_idx][np.array(topics_sample) == 14]
print(docs_topic14["raw_text"].head(30).tolist())

['Did you not see the boingy boingy at the end? ', '^ddddooooonnnntt ^sssssaaaaayyy ^sssswwwweeeeaaahhhhs', 'Baegitte', '~~MEKA~~ MAKO', 'he does not know da wae', 'Whoop de doo', 'Oink oink, motherfuckers.', 'Sargay', '', 'QUICK, WRITE ALL THE JAPANESE YOU KNOW\n\nNANI OMAE WA MOE SHINDEARU SHINE BAKA KAWAII DESU MUGIWARA KAIZOKU IKKEI THESE ARE ALL PROBABLY WRONG', '[*Gahwa* or *kawa*/*qahwa* is coffee.](https://en.wikipedia.org/wiki/Arabic_coffee)', 'YARRRR!', 'For the cooooooontent', 'REEEEEEE', "AND I'M JAVERE", 'OH SHIIEEET ', 'ho lee shit', 'NA Smoke. ', 'hnnnnnnnnnnnngggggggggggggg\n\n\n\ncaliente', 'Hmhmhmhmhmhmhehe', 'she should have even more *drool*', 'AuUGHGHGGH!\n', 'Craaaazy!', 'aMiazing?', 'ÜBERRASCHUNG! ', 'IS THIS EEEEEZZZZ MOOOODE?!', 'LADDAH', 'A better translation for "apagando las luces" would be "lights out".', '/r/oopsdidntmeanto', '[deleted]']


In [7]:
pd.set_option("display.max_rows", 250)
info = topic_model.get_topic_info()
print(info[["Topic", "Count", "Name"]])

     Topic   Count                                               Name
0       -1  144364                                  -1_to_the_and_you
1        0    7646                                0_mercy_rez_her_she
2        1    6933                            1_skins_skin_boxes_loot
3        2    6102                           2_chat_report_banned_ban
4        3    5881                            3_dps_tank_healer_tanks
5        4    4774                       4_hanzo_scatter_hanzos_arrow
6        5    3799               5_downvoted_opinion_downvotes_upvote
7        6    3750                          6_sombra_hack_sombras_her
8        7    3375                               7_dva_mech_dvas_meka
9        8    3241                       8_dallas_seoul_london_korean
10       9    3135                           9_read_sense_post_posted
11      10    2852                    10_lucio_lucios_lucioball_speed
12      11    2782                       11_genji_genjis_deflect_dash
13      12    2756  

Parece que en el cluster de ruido si que hemos perdido utterances con información acerca del target, pero esto se debe a que los embeddings de esas utterances simplemente no han encontrado una agrupación clara donde situarse. Esto no nos supone un gran problema ya que la idea principal es analizar los common topics populares dentro del subreddit.

Podemos observar que existen ciertos clusters como el 14, que son agrupaciones de vocabulario/memes/expresiones en diferentes idiomas, este cluster por ejemplo no tiene coherencia temática con Overwatch.

Pasa lo mismo en el cluster 5, donde principalmente se habla de upvotes o downvotes (vocabulario temático de la propia plataforma de reddit y sus metadatos) 

Otro hallazgo importante que si que tenemos que explorar mas a fondo son los siguientes clusters:
- 39_guidelineshttpswwwredditcomroverwatchwikiru...
- 111_submit_gameplay_reddit_website
- 112_submit_gameplay_reddit_website (nombre duplicado exacto)
- 132_submit_gameplay_reddit_website
- 168_submit_gameplay_reddit_website
- 131_submitting_please_such_automatically
- 186_submitting_please_such_automatically
- 215_submit_gameplay_removed_automatically
- 190_reddit_submit_gameplay_karma
- 116_karma_submission_reddit_required

Esto tiene pinta de ser texto de plantilla automático, probablemente de AutoModerator u otros bots populares de reddit que se encargan de administrar el subreddit de forma automatizada, poniendo textos repetidos en varios posts como las reglas de la comunidad o mensajes de hilos cerrados por spam.

Esto es un problema  de calidad de datos y filtrado.

In [8]:
# Verificar la hipótesis: buscar el patrón de texto boilerplate en el corpus completo
patron_bot = df_texts["raw_text"].str.contains(
    r"submit.*gameplay|guidelines.*reddit|automatically.*removed", case=False, na=False, regex=True
)
print(f"Utterances que matchean patrón de plantilla bot: {patron_bot.sum():,}")
print(df_texts.loc[patron_bot, "raw_text"].drop_duplicates().head(5).tolist())

Utterances que matchean patrón de plantilla bot: 45,649
["In accordance with /r/Overwatch's [Low-effort Guidelines](https://www.reddit.com/r/overwatch/wiki/rules), I am including additional context so that /u/AutoModerator does not remove this post.\n\nThe additional context is: Brigitte sometimes says this line at the start of a round. Given what we know of her and her background, what would she do with the extra time?", '# Introduction\n\nThis guide is an attempt to create a catalog of all of the general, unofficial rules that govern competitive Overwatch gameplay.  These rules are the principles developed over time by the best players.  They have learned to keep these in mind regardless of hero or role.  Mastery of the fundamentals are what makes a professional player so good, not complicated tricks for rare situations (though those help).  If you can keep all of these in mind or do them automatically during your ranked games you will be a better player by definition.\n\nI believe t

Efectivamente parecen mensajes auto generados típicos.

Vamos a cuantificar cuantos clusters hay con este tipo de información externa a nuestro target.

Aprovechando que ya generamos un documento con los términos del analisis chi2, vamos a usarlo como señal automática de relevancia por topic.

In [9]:
final_terms = pd.read_parquet("D:\\TFM\\data\\chi2_overwatch_terms_v2.parquet")
ow_vocab = set(final_terms.index)  # asumiendo que 'term' es el índice, ajusta si no

def relevance_score(topic_id, topic_model, ow_vocab, top_n=10):
    words = [w for w, _ in topic_model.get_topic(topic_id)[:top_n]]
    overlap = sum(1 for w in words if w in ow_vocab)
    return overlap / top_n

info = topic_model.get_topic_info()
info["ow_relevance"] = info["Topic"].apply(lambda t: relevance_score(t, topic_model, ow_vocab) if t != -1 else 0)

info_sorted = info.sort_values("ow_relevance")
print(info_sorted[["Topic", "Count", "Name", "ow_relevance"]].head(40))  # los de menor relevancia -> revisar primero

     Topic   Count                                               Name  \
0       -1  144364                                  -1_to_the_and_you   
17      16    2529                            16_fps_gpu_cpu_settings   
40      39    1603  39_guidelineshttpswwwredditcomroverwatchwikiru...   
60      59     967                     59_color_colorblind_colors_red   
48      47    1260                        47_joke_funny_sarcasm_laugh   
62      61     944                       61_beer_mcdonalds_drink_food   
46      45    1317                   45_english_swedish_accent_french   
112    111     446                 111_submit_gameplay_reddit_website   
96      95     544                        95_ampnbsp_post_youre_fonsi   
97      96     530                            96_music_song_dj_khaled   
111    110     451                         110_dad_daddy_mom_daughter   
104    103     468                               103_14_old_age_nudes   
124    123     406                           123_mo

In [ ]:
#Cargamos todos los datos por que reiniciamos el kernel
df_users = pd.read_parquet(r"D:\TFM\data\Overwatch.corpus\overwatch_clustering_dataset.parquet")
print(df_users["user"].value_counts().head(20))

user
[deleted]              222117
AutoModerator           57892
Kalranya                11644
IDontCareWhatsoever      8916
theodoreroberts          8493
SpriteGuy_000            7308
purewasted               5966
zumoro                   5132
dngrs                    4766
breedwell23              4556
Arokhantos               4139
Snow75                   4000
EYSHot01                 3943
whatisabaggins55         3823
TheMightyDontKneelM      3734
Maniak_                  3653
bmrtt                    3489
ltpirate                 3434
noobule                  3424
imnotjay2                3396
Name: count, dtype: int64


In [12]:
df_users.info()

<class 'pandas.DataFrame'>
RangeIndex: 3975655 entries, 0 to 3975654
Data columns (total 20 columns):
 #   Column               Dtype 
---  ------               ----- 
 0   id                   str   
 1   root                 str   
 2   reply_to             str   
 3   is_post              bool  
 4   user                 str   
 5   raw_text             str   
 6   text_for_clustering  str   
 7   post_title           str   
 8   timestamp            int64 
 9   score                int64 
 10  top_level_comment    str   
 11  gilded               int64 
 12  gildings             object
 13  subreddit            str   
 14  stickied             bool  
 15  permalink            str   
 16  author_flair_text    str   
 17  post_num_comments    int64 
 18  post_domain          str   
 19  clean_text           str   
dtypes: bool(2), int64(4), object(1), str(13)
memory usage: 3.2+ GB


Observamos que los mensajes de AuotModerator consituyen un 1.5% de nuestros datos, por lo que no es del todo necesario recalcular los embeddings. 

Si sumamos los counts de los topics que ya identificamos como sospechosos de bot (39, 111, 112, 132, 168, 215, 190, 186, 116...), nos da aproximadamente 4.300 documentos dentro de la muestra de 350k, es justo el orden de magnitud que esperarías si el 1.5% de AutoModerator en el corpus completo se traduce proporcionalmente a la muestra. Esto sugiere algo importante: el clustering ya está haciendo su trabajo correctamente.

De todas maneras merece la pena filtrar AutoModerator de la fuente de datos para el futuro, por si quisieramos repetir el pipeline, o para el paso de sentiment analysis.

In [15]:
n_automod = (df_users["user"] == "AutoModerator").sum()
print(f"Filas de AutoModerator en el dataset completo: {n_automod:,}")

df_clean_source = df_users[df_users["user"] != "AutoModerator"].copy()
df_clean_source.to_parquet("D:\\TFM\\data\\Overwatch.corpus\\overwatch_clustering_dataset_v2.parquet", index=False)

Filas de AutoModerator en el dataset completo: 57,892


Okay, ahora entramos en la parte de filtrado de los clusters que aportan valor de negocio y los que no.

Hemos observado que hay clusters (como los de bots) que claramente no tienen valor de negocio, sin embargo, otros clusters que tienen una puntuación muy baja en nuestros términos chi2, son sin embargo clusters bastante importantes (ej: cluster de performance del juego 'gpu, cpu, fps'). Por tanto, es una tarea complicada discernir con un simple algoritmo de reglas booleanas si ese cluster debería entrar en el reporte final o no.

Otro método posible para solucionar esto, es aprovechar que ya vamos a realizar una llamada a un LLM para resumir los clusters, usando las utterance que se encuentren cerca de los centroides, añadir en esa llamada una petición para que el LLM distinga entre si ese cluster es relevante o no.

Este método puede dar buen resultado, pero aqui se presenta otra hipotesis. Es posible que absolutamente todos los clusters tengan información valiosa para la toma de decisiones de negocio (exceptuando los de bots, que son un error de preprocesado de datos). Por ejemplo, hemos descartado antes el cluster de el vocabulario o 'slang' utilizado en el contexto del juego (memes, palabras de moda, etc). 

A simple vista este cluster no parece aportar mucho valor a la hora de tomar decisiones sobre el desarrollo y el mantenimiento del juego, pero realmente, este cluster de repertorio de expresiones populares entre los usuarios, si que podría tener un inmenso valor a la hora de lanzar una nueva campaña de marketing que conecte con los usuarios mas habituales del juego (o por contra partida que conecte con usuarios de la competencia). 

Tener esta agrupación de utterances tiene un inmenso valor, se podrían volver a aplicar técnicas de clusterizacion o TD-IDF dentro de este cluster de 'slang' para extraer cuales son los términos mas populares, mas de moda recientemente u otras métricas valiosas.

En resumen, es posible que una clusterización limpia con datos tratados de manera muy precisa, no necesite si quiera este filtrado.

Para comprobar esta hipotesis, vamos a generar un archivo con cada cluster y algunos textos significativos alrededor de sus centroides (no el centro puro, ya que esos textos a veces no son del todo representativos) para efectuar un etiquetado manual. 

Este etiquetado nos vuelve a servir como 'gold_set' también para evaluar que tal se comporta el filtrado con LLM.

In [2]:
import numpy as np
import pandas as pd

'''
Aqui nos encontramos un problema de memoria al intentar convertir una lista de strings de Python a
un array de NumPy, Numpy necesita un tipo de dato de ancho fijo, es decir busca el string mas largo
y reserva ese mismo espacio para todo el resto de elementos. Como tenemos posts bastante extensos,
esto es un problema. La solucion es no convertir a array de numpy, y trabajar con listas de Python,
 que son dinámicas.

def get_representative_samples(embeddings, cluster_labels, cluster_id, texts, n_samples=8, seed=42):
    mask = cluster_labels == cluster_id
    cluster_embeddings = embeddings[mask]
    cluster_texts = np.array(texts)[mask]

    if len(cluster_texts) == 0:
        return []

    centroid = cluster_embeddings.mean(axis=0)
    distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)

    n = min(n_samples, len(cluster_texts))
    closest_idx = np.argsort(distances)[:n]
    return cluster_texts[closest_idx].tolist()
'''

# 1. Cargar df_users (todas las features, dataset ORIGINAL sin filtrar) si no lo tienes ya en memoria
df_users = pd.read_parquet(r"D:\TFM\data\Overwatch.corpus\overwatch_clustering_dataset.parquet")

# 2. REALINEAR: forzar a que quede en el MISMO orden que ids/embeddings/topics_sample
df_users_aligned = df_users.set_index("id").loc[ids].reset_index()

# 3. Verificación de que la alineación es correcta antes de seguir
assert (df_users_aligned["id"].values == ids).all(), "¡Los ids no coinciden! No sigas sin arreglar esto."
print("Alineación correcta ✅")

# 4. Función de muestreo representativo (versión corregida, sin np.array sobre textos)
def get_representative_samples(embeddings, cluster_labels, cluster_id, texts, n_samples=8):
    idx = np.where(cluster_labels == cluster_id)[0]
    if len(idx) == 0:
        return []
    cluster_embeddings = embeddings[idx]
    centroid = cluster_embeddings.mean(axis=0)
    distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)
    n = min(n_samples, len(idx))
    closest_local_idx = np.argsort(distances)[:n]
    closest_global_idx = idx[closest_local_idx]
    return [texts[i] for i in closest_global_idx]  # texts es lista de Python, nunca np.array

# 5. Datos ya disponibles de la fase de fit, ahora todos alineados correctamente
emb_sample_arr = embeddings[sample_idx]
texts_sample_arr = df_users_aligned.iloc[sample_idx]["text_for_clustering"].fillna("").tolist()
topics_arr = np.array(topics_sample)

info = topic_model.get_topic_info()

# 6. Construcción de la tabla de revisión
rows = []
for topic_id in info["Topic"]:
    if topic_id == -1:
        continue  # el ruido -1 lo dejamos aparte, no tiene sentido revisarlo cluster a cluster

    keywords = [w for w, _ in topic_model.get_topic(topic_id)[:10]]
    samples = get_representative_samples(emb_sample_arr, topics_arr, topic_id, texts_sample_arr, n_samples=8)

    mask_topic = topics_arr == topic_id
    users_topic = df_users_aligned.iloc[sample_idx][mask_topic]["user"]
    pct_automod = (users_topic == "AutoModerator").mean()

    rows.append({
        "topic_id": topic_id,
        "count": info.loc[info["Topic"] == topic_id, "Count"].values[0],
        "keywords": ", ".join(keywords),
        "pct_automod": round(pct_automod, 3),
        "sample_docs": " ||| ".join(samples),
        "categoria_manual": "",  # bot_moderacion / meta_reddit / slang_comunidad / tema_juego / tema_comunidad / revisar
        "notas": "",
    })

df_review = pd.DataFrame(rows).sort_values("count", ascending=False)
df_review.to_csv("D:\\TFM\\data\\Overwatch.corpus\\topics_review_manual.csv", index=False, encoding="utf-8-sig")

print(f"\nTotal topics a revisar: {len(df_review)}")
print(f"(excluido el cluster -1 de ruido, {info.loc[info['Topic']==-1,'Count'].values[0]:,} docs)")

Alineación correcta ✅

Total topics a revisar: 226
(excluido el cluster -1 de ruido, 144,364 docs)


Dejamos un bloque para recargar variables para la proxima sesión de código. Los modelos y los embeddings quedan guardados en archivos

In [1]:
import numpy as np
import pandas as pd
from bertopic import BERTopic

BASE = r"D:\TFM\data\Overwatch.corpus"

# Embeddings + ids
embeddings = np.load(f"{BASE}\\overwatch_embeddings_full.npy")
ids = np.load(f"{BASE}\\overwatch_embeddings_ids.npy", allow_pickle=True)

# Modelo BERTopic ya entrenado
topic_model = BERTopic.load(f"{BASE}\\bertopic_model_350k")

# Muestra usada en el fit
sample_idx = np.load(f"{BASE}\\bertopic_sample_idx.npy")
topics_sample = np.load(f"{BASE}\\bertopic_sample_topics.npy")

# Metadata (user, raw_text, etc.), realineada igual que la sesión anterior
df_users = pd.read_parquet(f"{BASE}\\overwatch_clustering_dataset.parquet")
df_users_aligned = df_users.set_index("id").loc[ids].reset_index()
assert (df_users_aligned["id"].values == ids).all(), "Alineación rota, revisar antes de continuar"

# Info de topics y tu tabla de revisión ya generada
info = topic_model.get_topic_info()
df_review = pd.read_csv(f"{BASE}\\topics_review_manual.csv")

print("✅ Todo recargado y alineado. Listo para continuar la revisión manual.")
print(f"Topics: {len(info)-1} (+ruido). Filas en df_review: {len(df_review)}")

✅ Todo recargado y alineado. Listo para continuar la revisión manual.
Topics: 226 (+ruido). Filas en df_review: 226


Como ya hemos comentado, efectivamente hay muchos clusters que a pesar de no estar relacionados directamente con la toma de decisiones de desarrollo/mantenimiento del juego, contienen información valiosa para otros enfoques.

Para no perderlos simplemente de los clusters importantes, en la anotación manual quedan etiquetados como 'otros', de esta manera con filtrados lógicos quedarán correctamente etiquetados a continuación, a pesar de no entrar en el reporte final.

Esto que viene a continuación esta mas profundizado en la memoria, pero para no perder el hilo dentro del notebook hago un pequeño resumen.

Por la arquitectura decidida para hacer esta clusterizacion, algunas utterances quedan agrupadas en clusters que son principalmente de acuerdo / desacuerdo son su hilo superior. Estos clusters tienen un tamaño reducido, pero para no perder esta señal informativa, los vamos a dejar marcados como 'sentimiento' en la columna de categoria_manual.

De esta manera podrán ser reincorporados a su hilo superior en el analisis de sentimiento.

Por tanto, el etiquetado manual será el siguiente:

Tendremos dos columnas para anotar manualmente:
- categoria_manual 
- notas

Dentro de categoria_manual:
- juego -> todos los clusters con informacion directamente relevante para el juego
- descartar -> clusters con 0 información relevante (bots, ruido)
- otros -> clusters con información relevante para otros scopes (toxicidad, conductas humanas...)
- sentimiento -> clusters con señal de sentimiento para agrupar posteriormente

Ya tenemos categorizados los clusters, vamos a comprobar que hayan sido bien etiquetados

In [4]:
CATEGORIAS_VALIDAS = {"juego", "descartar", "otros", "sentimiento"}

df_review = pd.read_csv(r"D:\TFM\data\Overwatch.corpus\topics_reviewed_manual.csv", encoding="utf-8-sig")

# Normalizar espacios/mayúsculas por si acaso, antes de validar
df_review["categoria_manual"] = df_review["categoria_manual"].astype(str).str.strip().str.lower()

valores_unicos = set(df_review["categoria_manual"].unique())
valores_invalidos = valores_unicos - CATEGORIAS_VALIDAS

if valores_invalidos:
    print("⚠️ VALORES NO RECONOCIDOS ENCONTRADOS:")
    for val in valores_invalidos:
        filas_afectadas = df_review[df_review["categoria_manual"] == val]
        print(f"\n  '{val}' -> {len(filas_afectadas)} fila(s), topic_id: {filas_afectadas['topic_id'].tolist()}")
else:
    print("✅ Todos los valores de categoria_manual son válidos.")

print("\nDistribución de categorías:")
print(df_review["categoria_manual"].value_counts())

# Suma de control: debería coincidir con el total de topics revisados (226, sin contar el -1)
total_categorizado = df_review["categoria_manual"].isin(CATEGORIAS_VALIDAS).sum()
print(f"\nTotal categorizado correctamente: {total_categorizado} de {len(df_review)}")
if total_categorizado != len(df_review):
    print(f"⚠️ Faltan {len(df_review) - total_categorizado} filas por revisar o corregir")

✅ Todos los valores de categoria_manual son válidos.

Distribución de categorías:
categoria_manual
juego          162
sentimiento     34
descartar       16
otros           14
Name: count, dtype: int64

Total categorizado correctamente: 226 de 226


Parece que todo ha quedado bien categorizado.

El siguiente paso es proyectar el resto de embeddings para ver a cuales de estos clusters pertenecen.

UMAP tiene un método .transform() propio. Cuando ajustamos UMAP sobre la muestra de 350k, aprendió una estructura de vecinos y una proyeccion de 384 a 5 dimensiones. Para un punto nuevo, UMAP busca sus vecinos más cercanos dentro del conjunto de entrenamiento ya ajustado (no tiene que recalcular la estructura entera) y lo ubica en el espacio de 5 dimensiones de forma consistente.

HDBSCAN, gracias a que en el modelo activamos prediction_data=True, quedó guardado internamente la estructura necesaria para approximate_predict(). Dado un punto proyectado ya a 5 dimensiones, lo compora contra los puntos mas representativos por cada cluster aprendido, y le asigna el cluster mas cercano, o lo marca como ruido si no encaja.

topic_model.transform() encadena estos dos pasos.

Cabe mencionar que esto es una aproximación y no una reoptimización completa. Un punto asignado por transform() no tiene exactamente las mismas garantías estadísticas que uno que participó en el fit().

In [5]:
#Codigo para transform del resto del corpus, con el mismo patrón de chunking

import numpy as np
import os
import time

remaining_idx = np.setdiff1d(np.arange(len(embeddings)), sample_idx)
print(f"Documentos restantes a asignar: {len(remaining_idx):,}")

OUT_DIR = r"D:\TFM\data\Overwatch.corpus\transform_chunks"
os.makedirs(OUT_DIR, exist_ok=True)

BATCH = 150_000  # algo más conservador que en el embedding, porque transform es más costoso por doc
n_batches = (len(remaining_idx) // BATCH) + 1

for i in range(n_batches):
    start = i * BATCH
    end = min(start + BATCH, len(remaining_idx))
    if start >= len(remaining_idx):
        break

    out_path = f"{OUT_DIR}\\transform_chunk_{i:03d}.npz"
    if os.path.exists(out_path):
        print(f"Chunk {i} ya existe, saltando...")
        continue

    batch_idx = remaining_idx[start:end]
    batch_emb = embeddings[batch_idx]
    batch_docs = df_users_aligned.iloc[batch_idx]["text_for_clustering"].fillna("").tolist()

    t0 = time.time()
    batch_topics, _ = topic_model.transform(batch_docs, embeddings=batch_emb)
    elapsed = time.time() - t0

    np.savez(out_path, idx=batch_idx, topics=np.array(batch_topics))
    print(f"Chunk {i+1}/{n_batches} ({start:,}-{end:,}) -> {elapsed/60:.1f} min "
          f"({len(batch_idx)/elapsed:.1f} docs/seg)")

print("\n✅ Transform completado.")

Documentos restantes a asignar: 3,473,970
Chunk 1/24 (0-150,000) -> 4.2 min (592.1 docs/seg)
Chunk 2/24 (150,000-300,000) -> 3.6 min (689.1 docs/seg)
Chunk 3/24 (300,000-450,000) -> 3.7 min (681.2 docs/seg)
Chunk 4/24 (450,000-600,000) -> 3.8 min (651.0 docs/seg)
Chunk 5/24 (600,000-750,000) -> 3.3 min (756.4 docs/seg)
Chunk 6/24 (750,000-900,000) -> 3.3 min (762.1 docs/seg)
Chunk 7/24 (900,000-1,050,000) -> 3.2 min (779.1 docs/seg)
Chunk 8/24 (1,050,000-1,200,000) -> 3.3 min (760.7 docs/seg)
Chunk 9/24 (1,200,000-1,350,000) -> 3.3 min (763.0 docs/seg)
Chunk 10/24 (1,350,000-1,500,000) -> 3.3 min (762.2 docs/seg)
Chunk 11/24 (1,500,000-1,650,000) -> 3.2 min (789.4 docs/seg)
Chunk 12/24 (1,650,000-1,800,000) -> 3.3 min (767.8 docs/seg)
Chunk 13/24 (1,800,000-1,950,000) -> 3.2 min (769.7 docs/seg)
Chunk 14/24 (1,950,000-2,100,000) -> 3.3 min (768.0 docs/seg)
Chunk 15/24 (2,100,000-2,250,000) -> 3.3 min (766.4 docs/seg)
Chunk 16/24 (2,250,000-2,400,000) -> 3.3 min (751.1 docs/seg)
Chunk 1

In [6]:
#Consolidación de embeddings y clusters final total

import glob

chunk_files = sorted(glob.glob(f"{OUT_DIR}\\transform_chunk_*.npz"))

all_idx_transform = []
all_topics_transform = []
for f in chunk_files:
    data = np.load(f)
    all_idx_transform.append(data["idx"])
    all_topics_transform.append(data["topics"])

all_idx_transform = np.concatenate(all_idx_transform)
all_topics_transform = np.concatenate(all_topics_transform)

# Combinar con lo que ya teníamos del fit original (sample_idx + topics_sample)
final_idx = np.concatenate([sample_idx, all_idx_transform])
final_topics = np.concatenate([topics_sample, all_topics_transform])

df_topics_full = pd.DataFrame({"row_idx": final_idx, "topic": final_topics})
df_topics_full["id"] = ids[df_topics_full["row_idx"].values]

df_topics_full.to_parquet(r"D:\TFM\data\Overwatch.corpus\overwatch_all_topics.parquet", index=False)
print(f"Total documentos con topic asignado: {len(df_topics_full):,}")

Total documentos con topic asignado: 3,823,970


Perfecto, ya tenemos los 3.8 M de utterances clusterizadas, asi como sus clusters categorizados manualmente.

Ahora antes de automatizar la generación de un resumen de cada cluster, vamos a agrupar en un data set todas las utterances, con sus clusters, y a indexar también los clusters que vimos que realmente solo aportaban señal de sentimiento.

In [1]:
import pandas as pd
import numpy as np

# 1. Cargar topic asignado a cada documento
df_topics_full = pd.read_parquet(r"D:\TFM\data\Overwatch.corpus\overwatch_all_topics.parquet")

# 2. Traer metadata necesaria: reply_to (para la reasignación), raw_text, user, is_post, score, etc.
df_meta = pd.read_parquet(
    r"D:\TFM\data\Overwatch.corpus\overwatch_clustering_dataset.parquet",
    columns=["id", "reply_to", "root", "is_post", "raw_text", "user", "score", "post_title"]
)

df_full = df_topics_full.merge(df_meta, on="id", how="left")

# 3. Traer la categoría manual, indexada por topic_id
df_review = pd.read_csv(r"D:\TFM\data\Overwatch.corpus\topics_reviewed_manual.csv", encoding="utf-8-sig")
categoria_by_topic = dict(zip(df_review["topic_id"], df_review["categoria_manual"]))
categoria_by_topic[-1] = "descartar"  # el ruido -1 también se descarta, nunca se categorizó individualmente

df_full["categoria_manual"] = df_full["topic"].map(categoria_by_topic)

print(df_full["categoria_manual"].value_counts(dropna=False))
print(f"\nTotal filas: {len(df_full):,}")

categoria_manual
juego          1789474
descartar      1738750
sentimiento     167709
otros           128037
Name: count, dtype: int64

Total filas: 3,823,970


Reasignamos los cluster 'sentmiento' a su padre inmediato (hilo superior), si ese hilo superior es perteneciente a un cluster tematico del videojuego.

In [2]:
MAX_SALTOS = 3

topic_by_id = dict(zip(df_full["id"], df_full["topic"]))
categoria_by_id = dict(zip(df_full["id"], df_full["categoria_manual"]))
reply_to_by_id = dict(zip(df_full["id"], df_full["reply_to"]))

def resolver_topic_real(doc_id, max_saltos=MAX_SALTOS):
    current_id = doc_id
    for _ in range(max_saltos):
        parent_id = reply_to_by_id.get(current_id)
        if parent_id is None or pd.isna(parent_id):
            return -99  # es post raíz -> no resoluble
        parent_categoria = categoria_by_id.get(parent_id)
        if parent_categoria is None:
            return -99  # padre no encontrado en el corpus
        if parent_categoria != "sentimiento":
            return topic_by_id.get(parent_id)  # ancestro resuelto
        current_id = parent_id  # el padre también es 'sentimiento', sube un nivel más
    return -99  # límite de saltos alcanzado sin resolver

mask_sentimiento = df_full["categoria_manual"] == "sentimiento"
print(f"Documentos a resolver: {mask_sentimiento.sum():,}")

df_full["topic_final"] = df_full["topic"]
df_full.loc[mask_sentimiento, "topic_final"] = df_full.loc[mask_sentimiento, "id"].apply(resolver_topic_real)

# Estadísticas de resolución
resuelto = mask_sentimiento & (df_full["topic_final"] != -99)
no_resuelto = mask_sentimiento & (df_full["topic_final"] == -99)
print(f"Resueltos (reasignados a un cluster ancestro): {resuelto.sum():,}")
print(f"No resueltos (cadena de acuerdo sin ancla, o límite de saltos): {no_resuelto.sum():,}")

# Categoría final: la del topic_final ya resuelto (para los que se pudieron resolver)
df_full["categoria_final"] = df_full["topic_final"].map(categoria_by_topic)
df_full.loc[no_resuelto, "categoria_final"] = "descartar"  # sin ancla -> se descartan del análisis final

Documentos a resolver: 167,709
Resueltos (reasignados a un cluster ancestro): 156,543
No resueltos (cadena de acuerdo sin ancla, o límite de saltos): 11,166


Perfecto, hemos contextualizado de nuevo 156k de utterances de las 167k que quedaron mal clusterizadas. Esto es ideal para nuestro analisis de sentimiento posterior.

In [3]:
df_full.to_parquet(r"D:\TFM\data\Overwatch.corpus\overwatch_final_dataset.parquet", index=False)

print("\nDistribución final de categorías (tras resolución de 'sentimiento'):")
print(df_full["categoria_final"].value_counts())


Distribución final de categorías (tras resolución de 'sentimiento'):
categoria_final
juego        1870703
descartar    1820233
otros         133034
Name: count, dtype: int64


Mencionar que casi la mitad de utterances quedan descartadas. Pero analizando la clusterizacion sobre el sampleo de 350k utterances, vimos que 144k quedaron automaticamente descartadas como ruido en HBDSCAN, eso sumado a los clusters que contenian mensajes a descartar, hace que esta distribución parezca razonable.

Ahora ya tenemos guardado nuestro archivo 'overawtch_final_dataset.parquet'

En este archivo tenemos ya categorizada cada utterance con su topic_final (con los textos de sentimiento adjuntados a sus padres). Además categoria_final indica si entra al informe principal, al informe añadido de clusters informativos secundarios, o queda fuera.

In [4]:
df_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 3823970 entries, 0 to 3823969
Data columns (total 13 columns):
 #   Column            Dtype
---  ------            -----
 0   row_idx           int64
 1   topic             int64
 2   id                str  
 3   reply_to          str  
 4   root              str  
 5   is_post           bool 
 6   raw_text          str  
 7   user              str  
 8   score             int64
 9   post_title        str  
 10  categoria_manual  str  
 11  topic_final       int64
 12  categoria_final   str  
dtypes: bool(1), int64(4), str(8)
memory usage: 1.4 GB


Vamos ahora a generar resumenes de cada cluster, usando un LLM en local para los casos de uso que requieran mayor privacidad. 

Para esta tarea, vamos a hacer uso de la plataforma de ollama para ejecutar la tarea en local, usando el LLM europeo mistral.

Para que quede un poco documentado y sea reproducible, ahora tenemos que instalar ollama, y una vez instalado, desde la terminal tenemos que abrir un servidor local de ollama ejecutando 'ollama serve'. 

Esto nos deja listo el servidor en un puerto local. A continuación, desde otra nueva ventana cmd, escribimos 'ollama pull mistral' para descargar el modelo deseado.

Ahora vamos a hacer unas comprobaciones de que todo esta funcionando correctamente.

In [8]:
import requests
import json

# Test: ¿Está Ollama escuchando?
def test_ollama_connection(base_url="http://localhost:11434"):
    try:
        response = requests.get(f"{base_url}/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json().get("models", [])
            print("✅ Ollama conectado")
            print(f"Modelos disponibles: {len(models)}")
            for model in models:
                print(f"  - {model['name']}")
            return True
        else:
            print(f"❌ Ollama respondió con error: {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ No se puede conectar a Ollama: {e}")
        print("   ¿Ollama está corriendo? (ollama serve)")
        return False

test_ollama_connection()

✅ Ollama conectado
Modelos disponibles: 1
  - mistral:latest


True

In [9]:
import requests
import json

def test_generation(model="mistral"):
    """Test simple de generación"""
    prompt = "Resume en una frase: Overwatch es un videojuego shooter competitivo."
    
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "temperature": 0.3
        },
        timeout=60
    )
    
    if response.status_code == 200:
        result = response.json()
        print(f"✅ Generación exitosa")
        print(f"Respuesta: {result['response']}")
        print(f"Tokens generados: {result.get('eval_count', 'N/A')}")
        print(f"Tiempo: {result.get('total_duration', 'N/A')/1e9:.2f}s")
    else:
        print(f"❌ Error: {response.status_code}")
        print(response.text)

test_generation(model="mistral")

✅ Generación exitosa
Respuesta:  "Overwatch es un juego de disparos competitivo que ofrece a los jugadores el control de personajes únicos con habilidades especiales y un estilo visual llamativo."
Tokens generados: 45
Tiempo: 3.59s


Volvemos a cargar variables necesarias

In [32]:
import pandas as pd
import numpy as np
import requests
import time
from typing import List, Tuple

df_full = pd.read_parquet(r"D:\TFM\data\Overwatch.corpus\overwatch_final_dataset.parquet")
embeddings = np.load(r"D:\TFM\data\Overwatch.corpus\overwatch_embeddings_full.npy")
df_topics = pd.read_csv(r"D:\TFM\data\Overwatch.corpus\topics_reviewed_manual.csv", encoding="utf-8-sig")
df_topics.rename(columns={"count": "n_docs"}, inplace=True)

assert "row_idx" in df_full.columns, "Falta row_idx en df_full: sin esto no se puede localizar el embedding correcto"

print(f"✅ Datos cargados:")
print(f"   - Dataset principal: {len(df_full):,} utterances")
print(f"   - Embeddings: {embeddings.shape}")
print(f"   - Topics únicos: {len(df_topics)}")
print(df_topics["categoria_manual"].value_counts())

✅ Datos cargados:
   - Dataset principal: 3,823,970 utterances
   - Embeddings: (3823970, 384)
   - Topics únicos: 226
categoria_manual
juego          162
sentimiento     34
descartar       16
otros           14
Name: count, dtype: int64


Definimos la función para extraer samples que se encuentren alrededor de los centroides, pero no justo en ellos.

Aqui se ha corregido un bug bastante importante, y es que nuestra funcion get_centroid_samples estaba usando el indicie en el df de pandas, en lugar de la posición real en embeddings_arr.

Anteriormente se generó la columna row_idx precisamente para conservar esta información de la distancia de cada utterance al centroide de su cluster.

Para realmente crear un sampleo estructurado con utterances de diferentes distancias (60% centro + 40% periferia), necesitamos usar sus indices asignados dentro de cada cluster

In [ ]:
def get_centroid_samples(
    topic_id: int,
    df_data: pd.DataFrame,
    embeddings_arr: np.ndarray,
    n_samples: int = 5,
    strategy: str = "mixed"
) -> Tuple[List[str], List[int]]:
    """
    Extrae muestras representativas de un topic basadas en proximidad al centroide.
    Usa 'row_idx' (posición real en embeddings_arr), NO el índice de pandas de df_data.
    """
    mask = (df_data["categoria_final"] != "descartar") & (df_data["topic_final"] == topic_id)
    df_masked = df_data[mask]

    if len(df_masked) == 0:
        return [], []

    row_idx_vals = df_masked["row_idx"].values
    cluster_embeddings = embeddings_arr[row_idx_vals]

    centroid = cluster_embeddings.mean(axis=0)
    distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)

    n_samples = min(n_samples, len(df_masked))

    if strategy == "centroid":
        selected_local_idx = np.argsort(distances)[:n_samples]

    elif strategy == "diverse":
        percentiles = np.linspace(0, 100, n_samples + 1)
        selected_local_idx = []
        for i in range(n_samples):
            p_low, p_high = percentiles[i], percentiles[i + 1]
            candidates = np.where(
                (distances >= np.percentile(distances, p_low)) &
                (distances <= np.percentile(distances, p_high))
            )[0]
            if len(candidates) > 0:
                selected_local_idx.append(np.random.choice(candidates))
        selected_local_idx = np.array(selected_local_idx)

    elif strategy == "mixed":
        n_center = int(n_samples * 0.6)
        n_periph = n_samples - n_center
        center_local = np.argsort(distances)[:n_center]
        periph_local = np.argsort(distances)[-n_periph:]
        selected_local_idx = np.concatenate([center_local, periph_local])

    texts = df_masked.iloc[selected_local_idx]["raw_text"].fillna("").tolist()
    scores = df_masked.iloc[selected_local_idx]["score"].tolist()
    return texts, scores

# Test
sample_texts, sample_scores = get_centroid_samples(
    topic_id=0, df_data=df_full, embeddings_arr=embeddings, n_samples=5, strategy="mixed"
)
print(f"✅ Muestreo funcionando. Extrayendo {len(sample_texts)} samples del topic 0")

✅ Muestreo funcionando. Extrayendo 15 samples del topic 0


Ahora definimos una función para el prompt hacia mistral para que efectue los resumenes con el sampleo

In [28]:
def build_summary_prompt(topic_id: int, sample_texts: List[str], keywords_str: str, doc_count: int) -> str:
    samples_text = "\n\n".join([
        f"[Doc {i+1}] {txt[:200]}"
        for i, txt in enumerate(sample_texts[:5])
    ])

    prompt = f"""Topic #{topic_id} - Summarization Task

Keywords (theme indicators): {keywords_str}
Total documents in cluster: {doc_count:,}
Sample documents (n={len(sample_texts)}, a small non-representative sample out of {doc_count:,}):

{samples_text}

---
TASK: Write a SHORT 1-2 sentence description of WHAT TOPIC/ASPECT users are discussing here.

Instructions:
1. Focus strictly on the SUBJECT MATTER: which hero, mechanic, feature, or aspect of the game is being discussed.
2. Be SPECIFIC: mention hero names, mechanics, or features when relevant.
3. Avoid generic phrases like "players discussing the game".
4. Do NOT summarize overall sentiment, opinion polarity, or consensus (positive/negative/critical/praised).
   This sample of {len(sample_texts)} documents is too small to represent the sentiment distribution
   across the full {doc_count:,}-document cluster; sentiment is measured separately with a dedicated method.
5. If the sample docs happen to express an opinion, you may note the SPECIFIC POINT raised
   (e.g. "concerns about X mechanic"), but frame it as a topic present in the discussion,
   not as the general consensus of the community.

OUTPUT ONLY:
SUMMARY: [Your summary here]
CONFIDENCE: [high/medium/low]"""
    return prompt

In [ ]:
def summarize_topic_with_ollama(
    topic_id: int, keywords_str: str, doc_count: int,
    df_data: pd.DataFrame, embeddings_arr: np.ndarray, model_name: str = "mistral"
) -> dict:
    sample_texts, _ = get_centroid_samples(
        topic_id=topic_id, df_data=df_data, embeddings_arr=embeddings_arr,
        n_samples=15, strategy="mixed"
    )

    if not sample_texts:
        return {"summary": "Tópico vacío o sin documentos válidos", "confidence": "low"}

    prompt = build_summary_prompt(topic_id, sample_texts, keywords_str, doc_count)

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model_name,
                "prompt": prompt,
                "stream": False,
                "temperature": 0.2,
                "options": {"num_predict": 150}
            },
            timeout=90
        )

        if response.status_code != 200:
            return {"summary": f"Error Ollama Status {response.status_code}", "confidence": "error"}

        output_text = response.json().get("response", "").strip()

        summary, confidence = "N/A", "N/A"
        for line in output_text.split("\n"):
            line_stripped = line.strip()
            if line_stripped.upper().startswith("SUMMARY:"):
                summary = line_stripped.split(":", 1)[1].strip()
            elif line_stripped.upper().startswith("CONFIDENCE:"):
                confidence = line_stripped.split(":", 1)[1].strip()

        if summary == "N/A":
            summary = output_text  # fallback si el modelo no siguió el formato en absoluto

        return {"summary": summary, "confidence": confidence}

    except Exception as e:
        return {"summary": f"Error de conexión: {str(e)}", "confidence": "error"}

Y ahora ya generamos los resumenes y los guardamos

In [30]:
ai_summaries = []
ai_confidences = []

df_valid_topics = df_topics[df_topics["categoria_manual"] != "descartar"].copy()

print(f"🚀 Iniciando hidratación con Mistral para {len(df_valid_topics)} tópicos válidos...")
start_time = time.time()

for idx, row in df_valid_topics.iterrows():
    t_id = int(row["topic_id"])
    n_docs = int(row["n_docs"])
    keywords = row.get("keywords", "overwatch, game")  # <- corregido, antes era "words"

    print(f"Processing Topic #{t_id} ({n_docs} docs)...", end="\r")

    res = summarize_topic_with_ollama(
        topic_id=t_id, keywords_str=keywords, doc_count=n_docs,
        df_data=df_full, embeddings_arr=embeddings, model_name="mistral"
    )

    ai_summaries.append(res["summary"])
    ai_confidences.append(res["confidence"])
    time.sleep(0.1)

df_valid_topics["llm_summary"] = ai_summaries
df_valid_topics["llm_confidence"] = ai_confidences

elapsed = time.time() - start_time
print(f"\n\n✅ ¡Procesamiento completado en {elapsed/60:.2f} minutos!")

🚀 Iniciando hidratación con Mistral para 210 tópicos válidos...
Processing Topic #225 (205 docs)...

✅ ¡Procesamiento completado en 19.70 minutos!


In [31]:
# Hacer un merge match con el df_topics original para mantener los descartados con campos vacíos si es necesario
df_topics_final = df_topics.merge(
    df_valid_topics[["topic_id", "llm_summary", "llm_confidence"]], 
    on="topic_id", 
    how="left"
)

# Guardar a CSV
output_path = r"D:\TFM\data\Overwatch.corpus\topics_hydrated_mistral_v3.csv"
df_topics_final.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"💾 Resultados guardados con éxito en: {output_path}")

# Mostrar una muestra del resultado
df_topics_final[["topic_id", "categoria_manual", "llm_summary"]].dropna().head(10)

💾 Resultados guardados con éxito en: D:\TFM\data\Overwatch.corpus\topics_hydrated_mistral_v3.csv


,topic_id,categoria_manual,llm_summary
0,0,juego,Users are discussing the hero Mercy and her ga...
1,1,juego,Users are discussing purchasing skins or loot ...
2,2,juego,Users are discussing issues related to the rep...
3,3,juego,Users are discussing switching roles from Tank...
4,4,juego,Users are discussing the hero Hanzo in this cl...
5,5,otros,Users are discussing the issue of downvoting c...
6,6,juego,"Users are discussing Sombra, a hero in Overwat..."
7,7,juego,Users are discussing the hero D.va and her gam...
8,8,juego,The users are discussing matches and teams in ...
10,10,juego,Users are discussing the hero Lucio and his me...


Vamos a calcular algunas metricas de evaluacion de esta clusterizacion.

In [1]:
import numpy as np
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# 1. Definición de rutas identificadas en el notebook
PATH_EMBEDDINGS = r"D:\TFM\data\Overwatch.corpus\overwatch_embeddings_full.npy"
PATH_SAMPLE_IDX = r"D:\TFM\data\Overwatch.corpus\bertopic_sample_idx.npy"
PATH_SAMPLE_TOPICS = r"D:\TFM\data\Overwatch.corpus\bertopic_sample_topics.npy"

# 2. Carga de variables almacenadas
print("Cargando variables desde disco...")
all_embeddings = np.load(PATH_EMBEDDINGS)
sample_idx = np.load(PATH_SAMPLE_IDX)
topics_sample = np.load(PATH_SAMPLE_TOPICS)

# 3. Filtrar los embeddings correspondientes a la muestra usada en BERTopic
emb_sample = all_embeddings[sample_idx]

print(f"Forma del conjunto de embeddings muestreados: {emb_sample.shape}")
print(f"Total de asignaciones de topics cargadas: {len(topics_sample)}")

# 4. Filtrar los puntos marcados como Ruido (-1)
# HDBSCAN asigna -1 a los puntos que no pertenecen a ningún cluster denso
non_noise_mask = topics_sample != -1

X_eval = emb_sample[non_noise_mask]
labels_eval = topics_sample[non_noise_mask]

print(f"\nNúmero total de elementos evaluables (sin ruido -1): {len(labels_eval):,}")
print(f"Número de clusters únicos evaluados: {len(np.unique(labels_eval))}")

# ---------------------------------------------------------
# CALCULO DE MÉTRICAS
# ---------------------------------------------------------

# A. Davies-Bouldin e Índice Calinski-Harabasz (Rápidos y escalables)
print("\nCalculando Davies-Bouldin Index...")
db_index = davies_bouldin_score(X_eval, labels_eval)
print(f"-> Davies-Bouldin Index: {db_index:.4f} (Valores más cercanos a 0 indican mejor separación)")

print("\nCalculando Calinski-Harabasz Index...")
ch_index = calinski_harabasz_score(X_eval, labels_eval)
print(f"-> Calinski-Harabasz Index: {ch_index:.2f} (Valores más altos indican mejor separación)")

# B. Silhouette Score (Con subsampling seguro para evitar fugas de memoria RAM)
EVAL_SILHOUETTE_SAMPLE = 25_000  # Ajustable según la capacidad de tu RAM

if len(X_eval) > EVAL_SILHOUETTE_SAMPLE:
    rng = np.random.RandomState(42)
    sil_subsample_idx = rng.choice(len(X_eval), size=EVAL_SILHOUETTE_SAMPLE, replace=False)
    X_sil = X_eval[sil_subsample_idx]
    labels_sil = labels_eval[sil_subsample_idx]
else:
    X_sil = X_eval
    labels_sil = labels_eval

print(f"\nCalculando Silhouette Score sobre un subconjunto seguro de {len(labels_sil):,} puntos...")
# Se usa métrica 'cosine' porque los embeddings de BERT responden mejor a la similitud coseno
sil_score = silhouette_score(X_sil, labels_sil, metric='cosine')
print(f"-> Silhouette Score (Métrica Coseno): {sil_score:.4f} (Rango de -1 a 1, mayor es mejor)")

Cargando variables desde disco...
Forma del conjunto de embeddings muestreados: (350000, 384)
Total de asignaciones de topics cargadas: 350000

Número total de elementos evaluables (sin ruido -1): 205,636
Número de clusters únicos evaluados: 226

Calculando Davies-Bouldin Index...
-> Davies-Bouldin Index: 3.5003 (Valores más cercanos a 0 indican mejor separación)

Calculando Calinski-Harabasz Index...
-> Calinski-Harabasz Index: 335.41 (Valores más altos indican mejor separación)

Calculando Silhouette Score sobre un subconjunto seguro de 25,000 puntos...
-> Silhouette Score (Métrica Coseno): 0.0521 (Rango de -1 a 1, mayor es mejor)
